# WHO GHO OData usage walkthrough

This notebook shows how to explore the WHO GHO OData API, inspect dimensions, fetch indicator metadata, pull observation rows, and normalize them into dataframes for an ingestion pipeline.

Notes:
- The current API base used here is `https://ghoapi.azureedge.net/api`.
- WHO has published migration/deprecation notices around older data products, so treat this as a useful working interface, not a forever-stable contract.
- For production ingestion, persist the raw JSON row plus a typed projection.

In [1]:
from __future__ import annotations

import json
from urllib.parse import quote

import pandas as pd
import requests

In [2]:
BASE_URL = "https://ghoapi.azureedge.net/api"


def fetch_json(path: str, params: dict | None = None) -> dict:
    url = f"{BASE_URL}/{path.lstrip('/')}"
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()


def as_frame(payload: dict) -> pd.DataFrame:
    return pd.DataFrame(payload.get("value", []))

## 1. List available dimensions

Dimensions define the coded axes used in the observation rows, for example `COUNTRY`, `SEX`, `REGION`, and time-related dimensions.

In [3]:
dimensions = as_frame(fetch_json("Dimension"))
dimensions.head(20)

,Code,Title
0,ADVERTISINGTYPE,SUBSTANCE_ABUSE_ADVERTISING_TYPES
1,AGEGROUP,Age Group
2,ALCOHOLTYPE,Beverage Types
3,AMRGLASSCATEGORY,AMR GLASS Category
4,ANTIBIOTIC,Antibiotic
5,ARCHIVE,Archive date
6,ASSISTIVETECHBARRIER,Barriers to accessing assistive products
7,ASSISTIVETECHFUNDING,Funding for assistive tech products
8,ASSISTIVETECHPRODUCT,Assistive technology product
9,ASSISTIVETECHSATIACTIVITY,Satisfaction with assistive products for diffe...


## 2. Look up values for a specific dimension

Dimension values decode observation fields like `SpatialDim="USA"` or `Dim1="SEX_MLE"`.

In [4]:
country_values = as_frame(fetch_json("DIMENSION/COUNTRY/DimensionValues"))
country_values[["Code", "Title", "ParentDimension", "ParentCode"]].head(10)

,Code,Title,ParentDimension,ParentCode
0,ABW,Aruba,REGION,AMR
1,AFG,Afghanistan,REGION,EMR
2,AGO,Angola,REGION,AFR
3,AIA,Anguilla,REGION,AMR
4,ALB,Albania,REGION,EUR
5,AND,Andorra,REGION,EUR
6,ARE,United Arab Emirates,REGION,EMR
7,ARG,Argentina,REGION,AMR
8,ARM,Armenia,REGION,EUR
9,ASM,American Samoa,REGION,WPR


## 3. Find indicators you may want to ingest

The indicator registry is the main discovery surface. You can filter by code or name.

In [5]:
tb_indicators = as_frame(
    fetch_json(
        "Indicator",
        params={
            "$filter": "contains(IndicatorName,'tuberculosis')",
            "$top": 10,
        },
    )
)
tb_indicators[["IndicatorCode", "IndicatorName"]].head(10)

,IndicatorCode,IndicatorName
0,MDG_0000000017,Deaths due to tuberculosis among HIV-negative ...
1,MDG_0000000018,Deaths due to tuberculosis among HIV-positive ...
2,MDG_0000000020,Incidence of tuberculosis (per 100 000 populat...
3,MDG_0000000022,Tuberculosis detection rate under DOTS (%)
4,MDG_0000000023,Prevalence of tuberculosis (per 100 000 popula...
5,MDG_0000000024,Tuberculosis treatment success under DOTS (%)
6,MDG_0000000030,Smear-positive tuberculosis case-detection rat...
7,MDG_0000000031,Smear-positive tuberculosis treatment-success ...
8,TB_1,Tuberculosis treatment coverage
9,TB_c_lab_cul_5m,Laboratories providing tuberculosis diagnostic...


## 4. Inspect which dimensions an indicator uses

This is critical for normalization. `Dim1`, `Dim2`, and `Dim3` only make sense once you map them back to the dimension names allowed for that indicator.

In [6]:
indicator_code = "MDG_0000000020"
indicator_dimensions = as_frame(
    fetch_json(
        "IndicatorDimension",
        params={"$filter": f"IndicatorCode eq '{indicator_code}'"},
    )
)
indicator_dimensions

,IndicatorCode,Language,Dimension,DimensionName
0,MDG_0000000020,EN,WORLDBANKINCOMEGROUP,World Bank income group
1,MDG_0000000020,EN,COUNTRY,Country
2,MDG_0000000020,EN,REGION,WHO region
3,MDG_0000000020,EN,YEAR,Year
4,MDG_0000000020,EN,PUBLISHSTATE,Publish States


## 5. Pull actual observations for one indicator

This endpoint returns fact-like rows. Important common fields include:
- `IndicatorCode`
- `SpatialDim` and `SpatialDimType`
- `TimeDim`, `TimeDimensionBegin`, `TimeDimensionEnd`
- `Dim1...Dim3` and matching `Dim1Type...Dim3Type`
- `NumericValue`, `Low`, `High`, `Date`, `Comments`

In [7]:
observations = as_frame(
    fetch_json(
        indicator_code,
        params={
            "$filter": "SpatialDim eq 'USA'",
            "$orderby": "TimeDim desc",
            "$top": 5,
        },
    )
)
observations.T.head(30)

,0,1,2,3,4
Id,6253282,4788290,1260763,5014542,3812808
IndicatorCode,MDG_0000000020,MDG_0000000020,MDG_0000000020,MDG_0000000020,MDG_0000000020
SpatialDimType,COUNTRY,COUNTRY,COUNTRY,COUNTRY,COUNTRY
SpatialDim,USA,USA,USA,USA,USA
TimeDimType,YEAR,YEAR,YEAR,YEAR,YEAR
ParentLocationCode,AMR,AMR,AMR,AMR,AMR
ParentLocation,Americas,Americas,Americas,Americas,Americas
Dim1Type,None,None,None,None,None
TimeDim,2024,2023,2022,2021,2020
Dim1,None,None,None,None,None


## 6. Normalize a useful projection for ingestion

The safest MVP pattern is:
1. keep the full raw row as JSON
2. project a stable typed subset for analytics
3. decode dimensions in lookup tables

In [8]:
typed_projection = observations[
    [
        "IndicatorCode",
        "SpatialDim",
        "SpatialDimType",
        "TimeDim",
        "TimeDimensionBegin",
        "TimeDimensionEnd",
        "Dim1Type",
        "Dim1",
        "Dim2Type",
        "Dim2",
        "Dim3Type",
        "Dim3",
        "NumericValue",
        "Low",
        "High",
        "Date",
        "Comments",
    ]
].copy()

typed_projection["uncertainty_width"] = typed_projection["High"] - typed_projection["Low"]
typed_projection

,IndicatorCode,SpatialDim,SpatialDimType,TimeDim,TimeDimensionBegin,TimeDimensionEnd,Dim1Type,Dim1,Dim2Type,Dim2,Dim3Type,Dim3,NumericValue,Low,High,Date,Comments,uncertainty_width
0,MDG_0000000020,USA,COUNTRY,2024,2024-01-01T00:00:00+01:00,2024-12-31T00:00:00+01:00,None,None,None,None,None,None,3.2,2.6,3.9,2025-11-11T15:08:09.963+01:00,None,1.3
1,MDG_0000000020,USA,COUNTRY,2023,2023-01-01T00:00:00+01:00,2023-12-31T00:00:00+01:00,None,None,None,None,None,None,3.1,2.5,3.8,2025-11-11T15:08:09.963+01:00,None,1.3
2,MDG_0000000020,USA,COUNTRY,2022,2022-01-01T00:00:00+01:00,2022-12-31T00:00:00+01:00,None,None,None,None,None,None,2.6,2.1,3.2,2025-11-11T15:08:09.963+01:00,None,1.1
3,MDG_0000000020,USA,COUNTRY,2021,2021-01-01T00:00:00+01:00,2021-12-31T00:00:00+01:00,None,None,None,None,None,None,2.6,2.1,3.1,2025-11-11T15:08:09.963+01:00,None,1.0
4,MDG_0000000020,USA,COUNTRY,2020,2020-01-01T00:00:00+01:00,2020-12-31T00:00:00+01:00,None,None,None,None,None,None,2.4,1.9,2.8,2025-11-11T15:08:09.963+01:00,None,0.9


## 7. Decode coded dimension values

Here we join `SpatialDim` back to the `COUNTRY` dimension table. The same pattern applies to `SEX`, `REGION`, and other coded fields.

In [9]:
country_lookup = country_values[["Code", "Title"]].rename(
    columns={"Code": "SpatialDim", "Title": "country_name"}
)
decoded = typed_projection.merge(country_lookup, on="SpatialDim", how="left")
decoded[["IndicatorCode", "SpatialDim", "country_name", "TimeDim", "NumericValue", "Low", "High"]]

,IndicatorCode,SpatialDim,country_name,TimeDim,NumericValue,Low,High
0,MDG_0000000020,USA,United States of America,2024,3.2,2.6,3.9
1,MDG_0000000020,USA,United States of America,2023,3.1,2.5,3.8
2,MDG_0000000020,USA,United States of America,2022,2.6,2.1,3.2
3,MDG_0000000020,USA,United States of America,2021,2.6,2.1,3.1
4,MDG_0000000020,USA,United States of America,2020,2.4,1.9,2.8


## 8. Suggested canonical model for AIX

For this project, the useful logical tables are:
- `who_indicator`
- `who_dimension`
- `who_dimension_value`
- `who_indicator_dimension`
- `who_observation`

This lets you keep:
- discovery metadata
- dimension decoding
- indicator-specific dimensional context
- observation facts for scoring and downstream analysis

In [10]:
example_record = observations.iloc[0].to_dict()
print(json.dumps(example_record, indent=2)[:3000])

{
  "Id": 6253282,
  "IndicatorCode": "MDG_0000000020",
  "SpatialDimType": "COUNTRY",
  "SpatialDim": "USA",
  "TimeDimType": "YEAR",
  "ParentLocationCode": "AMR",
  "ParentLocation": "Americas",
  "Dim1Type": null,
  "TimeDim": 2024,
  "Dim1": null,
  "Dim2Type": null,
  "Dim2": null,
  "Dim3Type": null,
  "Dim3": null,
  "DataSourceDimType": null,
  "DataSourceDim": null,
  "Value": "3.2 [2.6-3.9]",
  "NumericValue": 3.2,
  "Low": 2.6,
  "High": 3.9,
  "Comments": null,
  "Date": "2025-11-11T15:08:09.963+01:00",
  "TimeDimensionValue": "2024",
  "TimeDimensionBegin": "2024-01-01T00:00:00+01:00",
  "TimeDimensionEnd": "2024-12-31T00:00:00+01:00"
}
